In [32]:
import os
from typing import Iterable
import altair as alt
import numpy as np
import pandas as pd
import requests
import eco_style
from pyproj import Transformer
from shapely.ops import unary_union
import geopandas as gpd
import json

alt.theme.enable("light")

ThemeRegistry.enable('light')

In [33]:
import geopandas

In [34]:
url = (
    "https://services1.arcgis.com/ESMARspQHYMw9BZ9/arcgis/rest/services/"
    "Wards_May_2024_Boundaries_UK_BSC/FeatureServer/0/query"
)

wigan_all_wards = (
    "'Abram','Ashton-in-Makerfield South','Bryn with Ashton-in-Makerfield North',"
    "'Hindley','Hindley Green','Orrell','Winstanley','Worsley Mesnes',"
    "'Aspull, New Springs & Whelley','Astley','Atherton North','Atherton South & Lilford',"
    "'Douglas','Golborne & Lowton West','Ince','Leigh Central & Higher Folds',"
    "'Leigh South','Leigh West','Lowton East','Pemberton',"
    "'Shevington with Lower Ground & Moor','Standish with Langtree',"
    "'Tyldesley & Mosley Common','Wigan Central','Wigan West'"
)

params_all = {
    "where": f"WD24NM IN ({wigan_all_wards})",
    "outFields": "WD24NM,WD24CD",
    "outSR": "4326",
    "f": "geojson"
}
wigan_geo_all = requests.get(url, params=params_all).json()
print(len(wigan_geo_all['features']), 'wards fetched')

25 wards fetched


In [35]:
df_raw = pd.read_csv("wigan_local_election_results_2026.csv", quotechar='"')

df_raw["Total_Votes"] = df_raw.groupby("Ward")["Votes"].transform("sum")
df_raw["Vote_Share_Pct"] = (df_raw["Votes"] / df_raw["Total_Votes"] * 100).round(1)

party_map = {
    "Reform UK":                                          "Reform UK",
    "Labour Party":                                       "Labour",
    "Labour and Co-operative Party":                      "Labour",
    "Green Party":                                        "Green",
    "The Green Party":                                    "Green",
    "Green Party-Save Our Green Space":                   "Green",
    "Conservative Party":                                 "Conservative",
    "The Conservative Party Candidate":                   "Conservative",
    "Liberal Democrats":                                  "Lib Dem",
    "Independent":                                        "Other",
    "Independent Network":                                "Other",
    "Leigh and Atherton Independent":                     "Other",
    "Leigh Central Independent":                          "Other",
    "Leigh South Independent":                            "Other",
    "Shevington Independents part of Wigan Independents": "Other",
    "Standish Independents":                              "Other",
    "Social Democratic Party":                            "Other",
}
df_raw["Party_Clean"] = df_raw["Party"].map(party_map).fillna("Other")

df_wide = df_raw.pivot_table(
    index="Ward", columns="Party_Clean", values="Vote_Share_Pct", aggfunc="sum"
).reset_index()
df_wide.columns.name = None

for col in ["Reform UK", "Labour", "Green", "Conservative", "Lib Dem", "Other"]:
    if col not in df_wide.columns:
        df_wide[col] = 0
df_wide = df_wide.fillna(0)

df_wide["winner"] = df_wide[
    ["Reform UK", "Labour", "Green", "Conservative", "Lib Dem", "Other"]
].idxmax(axis=1)
df_wide["WD24NM"] = df_wide["Ward"]


In [36]:
color_scale = alt.Scale(
    domain=["Reform UK", "Labour", "Green", "Conservative", "Lib Dem", "Other"],
    range=["#00bed6", "#E6224B", "#00A767", "#0063AF", "#F4C245", "#a8c0de"],
)

chart_map_wigan = (
    alt.Chart(alt.Data(values=wigan_geo_all["features"]))
    .mark_geoshape(stroke="white", strokeWidth=1.5)
    .transform_lookup(
        lookup="properties.WD24NM",
        from_=alt.LookupData(
            data=df_wide,
            key="WD24NM",
            fields=["Ward", "winner", "Reform UK", "Labour", "Green", "Conservative", "Lib Dem"],
        ),
    )
    .encode(
        color=alt.Color(
            "winner:N",
            scale=color_scale,
            legend=alt.Legend(
                title=None,
                orient="bottom",
                symbolType="square",
                symbolSize=120,
                labelFontSize=11,
            ),
        ),
        tooltip=[
            alt.Tooltip("Ward:N",          title="Ward"),
            alt.Tooltip("winner:N",        title="Winner"),
            alt.Tooltip("Reform UK:Q",     title="Reform UK (%)",    format=".1f"),
            alt.Tooltip("Labour:Q",        title="Labour (%)",       format=".1f"),
            alt.Tooltip("Green:Q",         title="Green (%)",        format=".1f"),
            alt.Tooltip("Conservative:Q",  title="Conservative (%)", format=".1f"),
            alt.Tooltip("Lib Dem:Q",       title="Lib Dem (%)",      format=".1f"),
        ],
    )
    .project(type="mercator")
    .properties(
        width=420,
        height=380,
        title=alt.TitleParams(
            text="Reform UK swept Wigan — except one ward",
            subtitle=["2026 local election results by ward, Wigan Council"],
            subtitleColor="#676A86",
            subtitleFontSize=12,
            anchor="start",
        ),
    )
)

chart_map_wigan

alt.Chart(...)

In [37]:
makerfield_wards = {
    'Abram', 'Ashton-in-Makerfield South', 'Bryn with Ashton-in-Makerfield North',
    'Hindley', 'Hindley Green', 'Orrell', 'Winstanley', 'Worsley Mesnes'
}

background = (
    alt.Chart(alt.Data(values=wigan_geo_all["features"]))
    .mark_geoshape(stroke="white", strokeWidth=1, opacity=0.25)
    .transform_lookup(
        lookup="properties.WD24NM",
        from_=alt.LookupData(
            data=df_wide,
            key="WD24NM",
            fields=["Ward", "winner", "Reform UK", "Labour", "Green", "Conservative", "Lib Dem"],
        ),
    )
    .encode(
        color=alt.Color("winner:N", scale=color_scale, legend=None),
        tooltip=[
            alt.Tooltip("Ward:N",         title="Ward"),
            alt.Tooltip("winner:N",       title="Winner"),
            alt.Tooltip("Reform UK:Q",    title="Reform UK (%)",    format=".1f"),
            alt.Tooltip("Labour:Q",       title="Labour (%)",       format=".1f"),
            alt.Tooltip("Green:Q",        title="Green (%)",        format=".1f"),
            alt.Tooltip("Conservative:Q", title="Conservative (%)", format=".1f"),
            alt.Tooltip("Lib Dem:Q",      title="Lib Dem (%)",      format=".1f"),
        ],
    )
    .project(type="mercator")
)

foreground = (
    alt.Chart(alt.Data(values=wigan_geo_all["features"]))
    .mark_geoshape(stroke="white", strokeWidth=1.5)
    .transform_filter(
        alt.FieldOneOfPredicate(
            field="properties.WD24NM",
            oneOf=list(makerfield_wards)
        )
    )
    .transform_lookup(
        lookup="properties.WD24NM",
        from_=alt.LookupData(
            data=df_wide,
            key="WD24NM",
            fields=["Ward", "winner", "Reform UK", "Labour", "Green", "Conservative", "Lib Dem"],
        ),
    )
    .encode(
        color=alt.Color(
            "winner:N",
            scale=color_scale,
            legend=alt.Legend(title=None, orient="bottom", symbolType="square",
                              symbolSize=120, labelFontSize=11),
        ),
        tooltip=[
            alt.Tooltip("Ward:N",         title="Ward"),
            alt.Tooltip("winner:N",       title="Winner"),
            alt.Tooltip("Reform UK:Q",    title="Reform UK (%)",    format=".1f"),
            alt.Tooltip("Labour:Q",       title="Labour (%)",       format=".1f"),
            alt.Tooltip("Green:Q",        title="Green (%)",        format=".1f"),
            alt.Tooltip("Conservative:Q", title="Conservative (%)", format=".1f"),
            alt.Tooltip("Lib Dem:Q",      title="Lib Dem (%)",      format=".1f"),
        ],
    )
    .project(type="mercator")
)

wigan_transparent = (
    (background + foreground)
    .properties(
        width=420,
        height=380,
        title=alt.TitleParams(
            text="Reform UK won 24 of 25 wards in Wigan's 2026 local elections",
            subtitle=["By-election constituency of Makerfield highlighted in full opacity"],
            subtitleColor="#676A86",
            subtitleFontSize=12,
            anchor="start",
        ),
    )
)

wigan_transparent

alt.LayerChart(...)

In [30]:
wigan_map = (
    alt.Chart(alt.Data(values=wigan_geo_all["features"]))
    .mark_geoshape(stroke="white", strokeWidth=1)
    .transform_lookup(
        lookup="properties.WD24NM",
        from_=alt.LookupData(
            data=df_wide,
            key="WD24NM",
            fields=["Ward", "Reform UK", "winner"],
        ),
    )
    .encode(
        color=alt.Color(
            "Reform UK:Q",
            scale=alt.Scale(
                domain=[10, 60],
                range=["#e8f7f8", "#00bed6"],
            ),
            legend=alt.Legend(
                title="Reform UK vote share (%)",
                orient="bottom",
                gradientLength=200,
            ),
        ),
        tooltip=[
            alt.Tooltip("Ward:N",         title="Ward"),
            alt.Tooltip("winner:N",       title="Winner"),
            alt.Tooltip("Reform UK:Q",    title="Reform UK (%)", format=".1f"),
        ],
    )
    .project(type="mercator")
    .properties(
        width=420,
        height=380,
        title=alt.TitleParams(
            text="Reform UK's vote share across Wigan, 2026",
            subtitle=["Darker shading = higher Reform UK vote share"],
            subtitleColor="#676A86",
            subtitleFontSize=12,
            anchor="start",
        ),
    )
)

wigan_map

alt.Chart(...)

In [38]:
gdf = gpd.GeoDataFrame.from_features(wigan_geo_all["features"], crs="EPSG:4326")

makerfield_gdf = gdf[gdf["WD24NM"].isin(makerfield_wards)]
makerfield_dissolved = makerfield_gdf.dissolve()

makerfield_outline = json.loads(makerfield_dissolved.to_json())

makerfield_border = (
    alt.Chart(alt.Data(values=makerfield_outline["features"]))
    .mark_geoshape(fill=None, stroke="#E6224B", strokeWidth=2.5)
    .project(type="mercator")
)

In [42]:
gdf = gpd.GeoDataFrame.from_features(wigan_geo_all["features"], crs="EPSG:4326")
makerfield_gdf = gdf[gdf["WD24NM"].isin(makerfield_wards)]
makerfield_dissolved = makerfield_gdf.dissolve()
makerfield_outline = json.loads(makerfield_dissolved.to_json())

# Layer 1 — Reform UK vote share choropleth
base = (
    alt.Chart(alt.Data(values=wigan_geo_all["features"]))
    .mark_geoshape(stroke="white", strokeWidth=1)
    .transform_lookup(
        lookup="properties.WD24NM",
        from_=alt.LookupData(
            data=df_wide,
            key="WD24NM",
            fields=["Ward", "winner", "Reform UK", "Labour", "Green", "Conservative", "Lib Dem"],
        ),
    )
    .encode(
        color=alt.Color(
            "Reform UK:Q",
            scale=alt.Scale(domain=[0, 60], range=["#e8f7f8", "#00bed6"]),
            legend=alt.Legend(title="Reform UK vote share (%)", orient="bottom", gradientLength=200),
        ),
        tooltip=[
            alt.Tooltip("Ward:N",         title="Ward"),
            alt.Tooltip("winner:N",       title="Winner"),
            alt.Tooltip("Reform UK:Q",    title="Reform UK (%)",    format=".1f"),
            alt.Tooltip("Labour:Q",       title="Labour (%)",       format=".1f"),
            alt.Tooltip("Green:Q",        title="Green (%)",        format=".1f"),
            alt.Tooltip("Conservative:Q", title="Conservative (%)", format=".1f"),
            alt.Tooltip("Lib Dem:Q",      title="Lib Dem (%)",      format=".1f"),
        ],
    )
    .project(type="mercator")
)

atherton_grey = (
    alt.Chart(alt.Data(values=wigan_geo_all["features"]))
    .mark_geoshape(fill="#d6d4d4", stroke="white", strokeWidth=1)
    .transform_filter(
        alt.FieldOneOfPredicate(field="properties.WD24NM", oneOf=["Atherton North"])
    )
    .transform_lookup(
        lookup="properties.WD24NM",
        from_=alt.LookupData(
            data=df_wide,
            key="WD24NM",
            fields=["Ward", "winner", "Reform UK", "Labour", "Green", "Conservative", "Lib Dem"],
        ),
    )
    .encode(
        tooltip=[
            alt.Tooltip("Ward:N",         title="Ward"),
            alt.Tooltip("winner:N",       title="Winner"),
            alt.Tooltip("Reform UK:Q",    title="Reform UK (%)",    format=".1f"),
            alt.Tooltip("Labour:Q",       title="Labour (%)",       format=".1f"),
            alt.Tooltip("Green:Q",        title="Green (%)",        format=".1f"),
            alt.Tooltip("Conservative:Q", title="Conservative (%)", format=".1f"),
            alt.Tooltip("Lib Dem:Q",      title="Lib Dem (%)",      format=".1f"),
        ],
    )
    .project(type="mercator")
)

# Layer 3 — dissolved Makerfield border in red
makerfield_border = (
    alt.Chart(alt.Data(values=makerfield_outline["features"]))
    .mark_geoshape(fill=None, stroke="#1E293B", strokeWidth=3)
    .project(type="mercator")
)

wigan_map = (
    (base + atherton_grey + makerfield_border)
    .properties(
        width=420,
        height=380,
        title=alt.TitleParams(
            text="Reform UK's vote share across Wigan Local Election, 2026",
            subtitle=[
                "Border = Makerfield by-election wards | Grey = Won by Independent",
                "Source: ECO Analysis of Wigan Local Election Results",
            ],
            subtitleColor="#676A86",
            subtitleFontSize=12,
            anchor="start",
        ),
    )
)

wigan_map

alt.LayerChart(...)

In [43]:
wigan_map.save("wigan_map.png", scale_factor=3.0)
wigan_map.save("wigan_map.json")

How will Burnham do? Local election results set the stage for a pivotal by-election in Makerfield, where Andy Burnham will stand for Labour. Reform UK won 24 of 25 wards in Wigan, with vote shares ranging from 29% in Atherton North (the only ward not won by Reform) to 56% in Abram. #ChartOfTheDay

Source: ECO Analysis of Wigan Local Election Results

Link: https://electionresults.wigan.gov.uk/LocalElections/Home/Index/19

🔗 Visit our Data Hub to explore and create your own charts. https://economicsobservatory.com/explore 